In [2]:
import pandas as pd
import numpy as np
import gc, os, warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

# ── UPDATE THESE PATHS ────────────────────────────────────────────────────────
RAW_DATA_PATH  = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\data\Jan - May '26 Data.csv"
OUTPUT_PATH    = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\retailer_day_features.parquet"
ENCODER_PATH   = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\label_encoders.pkl"
PROFILE_PATH   = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\retailer_profiles.parquet"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
print('Setup complete.')

Setup complete.


In [3]:
print('Loading raw data...')
df = pd.read_csv(RAW_DATA_PATH)
df['createdAt'] = pd.to_datetime(df['createdAt'], dayfirst=True)
print(f'Raw rows: {len(df):,}')

confirmed = df[df['orderStatus'].isin(['Delivered', 'PartiallyDelivered'])].copy()
orders = confirmed.drop_duplicates(subset='orderNumber')[[
    'orderNumber','customerId','createdAt',
    'hubName','shopType','retailerType','orderSource'
]].copy()

print(f'Confirmed unique orders : {len(orders):,}')
print(f'Unique retailers        : {orders["customerId"].nunique():,}')
print(f'Date range              : {orders["createdAt"].min().date()} → {orders["createdAt"].max().date()}')

del df, confirmed
gc.collect()
print('Memory freed.')

Loading raw data...
Raw rows: 609,723
Confirmed unique orders : 177,340
Unique retailers        : 8,640
Date range              : 2026-01-01 → 2026-05-31
Memory freed.


In [4]:
retailer_profile = orders.groupby('customerId').agg(
    hubName      = ('hubName',      lambda x: x.mode()[0]),
    shopType     = ('shopType',     lambda x: x.mode()[0]),
    retailerType = ('retailerType', lambda x: x.mode()[0] if x.notna().any() else 'Unknown'),
    app_orders   = ('orderSource',  lambda x: (x == 'App').sum()),
    call_orders  = ('orderSource',  lambda x: (x == 'CALLING_AGENT').sum()),
    total_orders = ('orderNumber',  'count'),
    first_order  = ('createdAt',    'min'),
    last_order   = ('createdAt',    'max'),
).reset_index()

retailer_profile['app_order_ratio'] = (
    retailer_profile['app_orders'] / retailer_profile['total_orders']
).round(4)

# Retailer tenure in days (how long they've been ordering)
retailer_profile['tenure_days'] = (
    retailer_profile['last_order'] - retailer_profile['first_order']
).dt.days

retailer_profile['retailerType'] = retailer_profile['retailerType'].fillna('Unknown')

print(f'Profile built for {len(retailer_profile):,} retailers')

# Save profile for use in app.py
retailer_profile.to_parquet(PROFILE_PATH, index=False)
print(f'Profile saved → {PROFILE_PATH}')

Profile built for 8,640 retailers
Profile saved → C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\retailer_profiles.parquet


In [5]:
all_retailers = orders['customerId'].unique()
all_dates     = pd.date_range(
    start=orders['createdAt'].min(),
    end=orders['createdAt'].max(),
    freq='D'
)

print(f'Retailers : {len(all_retailers):,}')
print(f'Days      : {len(all_dates)}')
print(f'Grid size : {len(all_retailers) * len(all_dates):,} rows')
print('Building grid...')

idx  = pd.MultiIndex.from_product([all_retailers, all_dates],
                                   names=['customerId','date'])
grid = pd.DataFrame(index=idx).reset_index()
print(f'Grid built: {len(grid):,} rows')

Retailers : 8,640
Days      : 151
Grid size : 1,304,640 rows
Building grid...
Grid built: 1,304,640 rows


In [6]:
order_flags = orders[['customerId','createdAt']].copy()
order_flags = order_flags.rename(columns={'createdAt':'date'})
order_flags['ordered_today'] = 1
order_flags = order_flags.drop_duplicates(subset=['customerId','date'])

grid = grid.merge(order_flags, on=['customerId','date'], how='left')
grid['ordered_today'] = grid['ordered_today'].fillna(0).astype(int)

print(f'Total rows : {len(grid):,}')
print(f'Positive   : {grid["ordered_today"].sum():,} ({grid["ordered_today"].mean()*100:.1f}%)')
print(f'Negative   : {(grid["ordered_today"]==0).sum():,}')

Total rows : 1,304,640
Positive   : 166,864 (12.8%)
Negative   : 1,137,776


In [7]:
print('Sorting grid...')
grid = grid.sort_values(['customerId','date']).reset_index(drop=True)
grid['_ord'] = grid['ordered_today']

print('Computing rolling order counts (shift=1 for no leakage)...')
grp = grid.groupby('customerId')['_ord']

grid['orders_last_3_days']  = grp.transform(lambda x: x.shift(1).rolling(3,  min_periods=1).sum())
grid['orders_last_7_days']  = grp.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).sum())
grid['orders_last_14_days'] = grp.transform(lambda x: x.shift(1).rolling(14, min_periods=1).sum())
grid['orders_last_30_days'] = grp.transform(lambda x: x.shift(1).rolling(30, min_periods=1).sum())
grid['total_orders_so_far'] = grp.transform(lambda x: x.shift(1).expanding().sum())

print('Rolling counts done.')

Sorting grid...
Computing rolling order counts (shift=1 for no leakage)...
Rolling counts done.


In [8]:
print('Computing days since last order...')

grid['_order_date']    = grid['date'].where(grid['_ord'] == 1)
grid['last_order_date'] = grid.groupby('customerId')['_order_date'].transform(
    lambda x: x.shift(1).ffill()
)
grid['days_since_last_order'] = (grid['date'] - grid['last_order_date']).dt.days
grid['days_since_last_order'] = grid['days_since_last_order'].fillna(999)

print('days_since_last_order done.')

Computing days since last order...
days_since_last_order done.


In [9]:
print('Computing inter-order gap statistics...')

orders_sorted = orders.sort_values(['customerId','createdAt'])
orders_sorted['gap'] = orders_sorted.groupby('customerId')['createdAt'].diff().dt.days

gap_stats = orders_sorted.groupby('customerId')['gap'].agg(
    avg_gap_between_orders = 'mean',
    std_gap_between_orders = 'std',
    median_gap             = 'median',
).reset_index()

gap_stats['avg_gap_between_orders'] = gap_stats['avg_gap_between_orders'].fillna(30).round(2)
gap_stats['std_gap_between_orders'] = gap_stats['std_gap_between_orders'].fillna(0).round(2)
gap_stats['median_gap']             = gap_stats['median_gap'].fillna(30).round(2)

grid = grid.merge(gap_stats, on='customerId', how='left')
print('Gap stats done.')

Computing inter-order gap statistics...
Gap stats done.


In [10]:
print('Computing overdue and regularity features...')

grid['expected_next_order'] = (
    grid['last_order_date'] +
    pd.to_timedelta(grid['avg_gap_between_orders'], unit='D')
)
grid['days_overdue']      = (grid['date'] - grid['expected_next_order']).dt.days
grid['days_overdue']      = grid['days_overdue'].fillna(0)
grid['is_overdue']        = (grid['days_overdue'] > 0).astype(int)
grid['order_regularity']  = 1 / (grid['std_gap_between_orders'] + 1)

# Days since last order relative to their personal avg gap
# >1 means they are overdue relative to their own pattern
grid['overdue_ratio'] = (
    grid['days_since_last_order'] / (grid['avg_gap_between_orders'] + 1)
).clip(upper=10).round(3)

print('Overdue features done.')

Computing overdue and regularity features...
Overdue features done.


In [11]:
print('Adding temporal features...')

grid['day_of_week']    = grid['date'].dt.dayofweek   # 0=Mon, 6=Sun
grid['day_of_month']   = grid['date'].dt.day
grid['week_of_month']  = (grid['date'].dt.day - 1) // 7 + 1
grid['month']          = grid['date'].dt.month
grid['is_weekend']     = (grid['date'].dt.dayofweek >= 5).astype(int)
grid['is_month_start'] = (grid['date'].dt.day <= 3).astype(int)
grid['is_month_end']   = (grid['date'].dt.day >= 28).astype(int)

print('Temporal features done.')

Adding temporal features...
Temporal features done.


In [12]:
print('Merging retailer profile...')
grid = grid.merge(
    retailer_profile[['customerId','hubName','shopType','retailerType',
                       'app_order_ratio','tenure_days']],
    on='customerId', how='left'
)
grid['retailerType'] = grid['retailerType'].fillna('Unknown')
print(f'Grid shape after merge: {grid.shape}')

Merging retailer profile...
Grid shape after merge: (1304640, 32)


In [13]:
import pickle

cat_cols = ['hubName', 'shopType', 'retailerType']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    grid[col + '_enc'] = le.fit_transform(grid[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {grid[col].nunique()} categories → encoded')

# Save encoders — needed in Notebook 4 when predicting June rows
with open(ENCODER_PATH, 'wb') as f:
    pickle.dump(label_encoders, f)
print(f'\nEncoders saved → {ENCODER_PATH}')

hubName: 11 categories → encoded
shopType: 10 categories → encoded
retailerType: 4 categories → encoded

Encoders saved → C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\label_encoders.pkl


In [14]:
drop_cols = ['_ord','_order_date','last_order_date','expected_next_order']
grid = grid.drop(columns=[c for c in drop_cols if c in grid.columns])

FEATURE_COLS = [
    'days_since_last_order',
    'avg_gap_between_orders',
    'std_gap_between_orders',
    'median_gap',
    'orders_last_3_days',
    'orders_last_7_days',
    'orders_last_14_days',
    'orders_last_30_days',
    'total_orders_so_far',
    'days_overdue',
    'is_overdue',
    'order_regularity',
    'overdue_ratio',
    'app_order_ratio',
    'tenure_days',
    'day_of_week',
    'day_of_month',
    'week_of_month',
    'month',
    'is_weekend',
    'is_month_start',
    'is_month_end',
    'hubName_enc',
    'shopType_enc',
    'retailerType_enc',
]

# Null check
null_check = grid[FEATURE_COLS].isnull().sum()
bad = null_check[null_check > 0]
if len(bad) > 0:
    print('Nulls found — filling with 0:')
    print(bad)
    grid[FEATURE_COLS] = grid[FEATURE_COLS].fillna(0)
else:
    print('No nulls in feature columns. Clean.')

print(f'\nFinal grid: {grid.shape[0]:,} rows × {grid.shape[1]} columns')
print(f'Features  : {len(FEATURE_COLS)}')

Nulls found — filling with 0:
orders_last_3_days     8640
orders_last_7_days     8640
orders_last_14_days    8640
orders_last_30_days    8640
total_orders_so_far    8640
dtype: int64

Final grid: 1,304,640 rows × 31 columns
Features  : 25


In [15]:
print('Saving to parquet...')
grid.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved → {OUTPUT_PATH}')
print(f'File size: {os.path.getsize(OUTPUT_PATH)/(1024**2):.1f} MB')
print()
print('=== FEATURE ENGINEERING COMPLETE ===')
print(f'Positive (ordered=1): {grid["ordered_today"].sum():,} ({grid["ordered_today"].mean()*100:.1f}%)')
print('\nNext → Run Notebook 3 (Model Training)')

Saving to parquet...
Saved → C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\retailer_day_features.parquet
File size: 4.5 MB

=== FEATURE ENGINEERING COMPLETE ===
Positive (ordered=1): 166,864 (12.8%)

Next → Run Notebook 3 (Model Training)


In [16]:
grid = pd.read_parquet(
    "../processed/retailer_day_features.parquet"
)

print(grid.shape)

(1304640, 31)
